# Customer Churn Analysis and Business Intelligence
### Sreenidhi Institute of Science and Technology (SNIST)
**Department:** Computer Science and Engineering — Data Science
**Team:** G Mythili (24311A67H8), K Ashiwith (24311A67H6), D Niyati (24311A67H7)

This notebook demonstrates the complete end-to-end data processing, exploratory data analysis, inferential statistics, and SQL querying.

In [1]:
import pandas as pd
import numpy as np
import sqlite3

# 1. Load Dataset
df = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')
print('Raw shape:', df.shape)
df.head()

In [2]:
# 2. Clean TotalCharges whitespace
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce').fillna(0.0)
df['SeniorCitizen'] = df['SeniorCitizen'].astype(int)
print('Missing values after cleaning:\n', df.isnull().sum()[df.isnull().sum() > 0])

In [3]:
# 3. Key Retention Statistics
total = len(df)
churned = (df['Churn'] == 'Yes').sum()
rate = (churned / total) * 100
print(f'Total Customers: {total}')
print(f'Churned Customers: {churned}')
print(f'Overall Churn Rate: {rate:.2f}%')
print(f'Average Monthly Charges: ${df["MonthlyCharges"].mean():.2f}')

In [4]:
# 4. Contract-wise Churn Analysis
contract_summary = df.groupby('Contract').agg(
    Total=('customerID', 'count'),
    Churned=('Churn', lambda x: (x == 'Yes').sum()),
    Churn_Rate_Pct=('Churn', lambda x: round((x == 'Yes').mean() * 100, 2))
).reset_index()
contract_summary

In [5]:
# 5. SQLite Relational Query Execution
conn = sqlite3.connect('../database/telco_churn.db')
sql_q = '''
SELECT PaymentMethod, COUNT(*) as Total, SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) as Churned,
ROUND(100.0 * SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END)/COUNT(*), 2) as ChurnRatePct
FROM telco_churn GROUP BY PaymentMethod ORDER BY ChurnRatePct DESC;
'''
sql_res = pd.read_sql_query(sql_q, conn)
conn.close()
sql_res